# PCL detection: batch size comparison (16, 32, 64)

RoBERTa-base with HTML stripping; class-weighted CE loss only. This notebook trains one model per batch size (16, 32, 64) and compares dev metrics.

In [1]:
import html
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256
BATCH_SIZES = [16, 32, 64]  # compare these batch sizes
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
USE_CLASS_WEIGHTS = True  # class-weighted CE loss only

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Resolve project root regardless of whether notebook is run from project root or src/
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing data/ directory.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

/vol/bitbucket/tq422/nlpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Project root: /vol/bitbucket/tq422/NLP_cw


## Load data and build train/dev splits

In [2]:
df = pd.read_csv(
    RAW_DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
)

df["label_binary"] = df["label"].apply(lambda x: 0 if x in [0, 1] else 1)

# Split IDs (match organizer: exactly one row per ID from split files)
a_train = pd.read_csv(RAW_DATA_DIR / "train_semeval_parids-labels.csv")
a_dev = pd.read_csv(RAW_DATA_DIR / "dev_semeval_parids-labels.csv")

df_sub = df[["par_id", "text", "label_binary"]].drop_duplicates(subset=["par_id"], keep="first").copy()
df_sub["text"] = df_sub["text"].fillna("").astype(str)

train_df = a_train[["par_id"]].merge(df_sub, on="par_id", how="left")
dev_df = a_dev[["par_id"]].merge(df_sub, on="par_id", how="left")
train_df["text"] = train_df["text"].fillna("").astype(str)
dev_df["text"] = dev_df["text"].fillna("").astype(str)
train_df["label_binary"] = train_df["label_binary"].fillna(0).astype(int)
dev_df["label_binary"] = dev_df["label_binary"].fillna(0).astype(int)

# Split a validation set from training data; dev is held out and not used during training
VAL_RATIO = 0.1
train_df, val_df = train_test_split(
    train_df, test_size=VAL_RATIO, stratify=train_df["label_binary"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Dev size:   {len(dev_df)} (held out, not used during training)")
print("\nTrain label distribution:")
print(train_df["label_binary"].value_counts())
print("\nVal label distribution:")
print(val_df["label_binary"].value_counts())
print("\nDev label distribution:")
print(dev_df["label_binary"].value_counts())

Train size: 7537
Val size:   838
Dev size:   2094 (held out, not used during training)

Train label distribution:
label_binary
0    6822
1     715
Name: count, dtype: int64

Val label distribution:
label_binary
0    759
1     79
Name: count, dtype: int64

Dev label distribution:
label_binary
0    1895
1     199
Name: count, dtype: int64


## Preprocess text: remove HTML markups

From data exploration: texts can contain HTML tags (e.g. `<h>`, `</p>`) that do not add useful signal. Strip tags and unescape HTML entities, then normalize whitespace.

In [3]:
def strip_html_and_clean(text: str) -> str:
    """Remove HTML tags, unescape entities, and normalize whitespace."""
    if not isinstance(text, str) or not text.strip():
        return text.strip() if isinstance(text, str) else ""
    # Remove HTML tags (pattern from data_exploration: <[^>]+>)
    no_tags = re.sub(r"<[^>]+>", "", text)
    # Unescape HTML entities (e.g. &amp; -> &, &lt; -> <)
    unescaped = html.unescape(no_tags)
    # Collapse multiple spaces and strip
    return re.sub(r"\s+", " ", unescaped).strip()


train_df["text"] = train_df["text"].apply(strip_html_and_clean)
val_df["text"] = val_df["text"].apply(strip_html_and_clean)
dev_df["text"] = dev_df["text"].apply(strip_html_and_clean)

# Sanity check: ensure no empty texts
train_df["text"] = train_df["text"].replace("", " ")
val_df["text"] = val_df["text"].replace("", " ")
dev_df["text"] = dev_df["text"].replace("", " ")
print("HTML preprocessing applied to train, val, and dev texts.")

HTML preprocessing applied to train, val, and dev texts.


In [4]:
# Class weights for loss only (inverse frequency; no sampling)
n0 = (train_df["label_binary"] == 0).sum()
n1 = (train_df["label_binary"] == 1).sum()
n = n0 + n1
w0 = n / (2 * n0)
w1 = n / (2 * n1)
class_weights = torch.tensor([w0, w1], dtype=torch.float32)

print(f"Class weights (0, 1): {class_weights.tolist()}")

Class weights (0, 1): [0.55240398645401, 5.270629405975342]


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = PCLDataset(train_df["text"], train_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
val_dataset = PCLDataset(val_df["text"], val_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
dev_dataset = PCLDataset(dev_df["text"], dev_df["label_binary"], tokenizer, max_length=MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer and datasets ready.")

Tokenizer and datasets ready.


## Training

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average=None, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = (preds == labels).mean()

    return {
        "accuracy": float(acc),
        "precision_pos": float(p_pos[0]),
        "recall_pos": float(r_pos[0]),
        "f1_pos": float(f1_pos[0]),
        "f1_macro": float(f1_macro),
    }


class ImbalanceTrainer(Trainer):
    """Trainer with class-weighted CE loss only (no sampling)."""

    def __init__(self, class_weights=None, use_class_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
        self.use_class_weights = use_class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.use_class_weights and self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


import json
results_per_bs = []

for bs in BATCH_SIZES:
    print(f"\n{'='*60}\nBatch size: {bs}\n{'='*60}")
    MODEL_DIR = PROJECT_ROOT / "models" / f"roberta_pcl_imbalance_weights_only_bs{bs}"
    OUTPUT_PREFIX = f"roberta_pcl_imbalance_weights_only_bs{bs}"
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    training_args = TrainingArguments(
        output_dir=str(MODEL_DIR / "checkpoints"),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1_pos",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
    )

    trainer = ImbalanceTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
        class_weights=class_weights if USE_CLASS_WEIGHTS else None,
        use_class_weights=USE_CLASS_WEIGHTS,
    )

    train_result = trainer.train()
    eval_metrics = trainer.evaluate(dev_dataset)

    pred_output = trainer.predict(dev_dataset)
    dev_preds = np.argmax(pred_output.predictions, axis=-1)
    dev_labels = pred_output.label_ids
    report_str = classification_report(dev_labels, dev_preds, target_names=["No PCL", "PCL"], digits=4, zero_division=0)

    record = {"batch_size": bs, "eval_loss": eval_metrics.get("eval_loss"), "eval_accuracy": eval_metrics.get("eval_accuracy"),
              "eval_precision_pos": eval_metrics.get("eval_precision_pos"), "eval_recall_pos": eval_metrics.get("eval_recall_pos"),
              "eval_f1_pos": eval_metrics.get("eval_f1_pos"), "eval_f1_macro": eval_metrics.get("eval_f1_macro"),
              "classification_report": report_str}
    results_per_bs.append(record)

    dev_metrics_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_dev_metrics.json"
    with open(dev_metrics_path, "w", encoding="utf-8") as f:
        json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in record.items()}, f, indent=2)
    trainer.save_model(str(MODEL_DIR))
    tokenizer.save_pretrained(str(MODEL_DIR))
    print(f"  Saved model to {MODEL_DIR}; dev f1_pos={record['eval_f1_pos']:.4f}")

print("\nTraining and evaluation finished for all batch sizes.")


Batch size: 16


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.608300,0.486983,0.781623,0.275862,0.810127,0.411576,0.638755
2,0.473700,0.639064,0.912888,0.531250,0.645570,0.582857,0.767111
3,0.305300,1.230839,0.914081,0.546667,0.518987,0.532468,0.742581
4,0.158000,1.323900,0.900955,0.480000,0.607595,0.536313,0.740434
5,0.201500,1.449481,0.909308,0.518519,0.531646,0.525000,0.737434


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Saved model to /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_imbalance_weights_only_bs16; dev f1_pos=0.5701

Batch size: 32


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.513700,0.597205,0.607399,0.189055,0.962025,0.316008,0.520347
2,0.438300,0.567102,0.891408,0.450000,0.683544,0.542714,0.740551
3,0.266400,0.874256,0.916468,0.561644,0.518987,0.539474,0.746771
4,0.299100,1.130935,0.917661,0.558140,0.607595,0.581818,0.768077
5,0.104400,1.516968,0.922434,0.620690,0.455696,0.525547,0.741656
6,0.073900,1.825771,0.918854,0.593220,0.443038,0.507246,0.731517
7,0.063400,2.087821,0.921241,0.618182,0.430380,0.507463,0.732331


  Saved model to /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_imbalance_weights_only_bs32; dev f1_pos=0.6074

Batch size: 64


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.591400,0.456288,0.805489,0.296117,0.772152,0.428070,0.655444
2,0.415000,0.508269,0.861575,0.375839,0.708861,0.491228,0.705559
3,0.276600,0.585513,0.878282,0.412214,0.683544,0.514286,0.722354
4,0.182800,0.793240,0.908115,0.511111,0.582278,0.544379,0.746642
5,0.167600,1.087677,0.909308,0.518987,0.518987,0.518987,0.734461
6,0.077200,1.173907,0.891408,0.444444,0.607595,0.513369,0.726127
7,0.025600,1.702770,0.900955,0.473684,0.455696,0.464516,0.704973


  Saved model to /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_imbalance_weights_only_bs64; dev f1_pos=0.5848

Training and evaluation finished for all batch sizes.


## Batch size comparison

Summary of dev-set metrics for each batch size. Best batch size is chosen by **f1_pos** (positive class F1).

In [7]:
# Compare dev metrics across batch sizes
comparison_df = pd.DataFrame([
    {
        "batch_size": r["batch_size"],
        "eval_loss": r["eval_loss"],
        "accuracy": r["eval_accuracy"],
        "precision_pos": r["eval_precision_pos"],
        "recall_pos": r["eval_recall_pos"],
        "f1_pos": r["eval_f1_pos"],
        "f1_macro": r["eval_f1_macro"],
    }
    for r in results_per_bs
])
comparison_df = comparison_df.set_index("batch_size")
print("Dev set comparison (batch size 16, 32, 64):")
display(comparison_df)

comparison_path = OUTPUT_DIR / "pcl_imbalance_batch_size_comparison.csv"
comparison_df.to_csv(comparison_path)
print(f"\nSaved comparison to {comparison_path}")
best_bs = max(results_per_bs, key=lambda r: r["eval_f1_pos"])["batch_size"]
print(f"Best batch size by dev f1_pos: {best_bs}")

Dev set comparison (batch size 16, 32, 64):


,eval_loss,accuracy,precision_pos,recall_pos,f1_pos,f1_macro
batch_size,,,,,,
16,0.226424,0.913563,0.540541,0.603015,0.570071,0.761011
32,0.325785,0.924069,0.597087,0.618090,0.607407,0.782689
64,0.261466,0.911175,0.526104,0.658291,0.584821,0.767544



Saved comparison to /vol/bitbucket/tq422/NLP_cw/output/pcl_imbalance_batch_size_comparison.csv
Best batch size by dev f1_pos: 32


In [8]:
# Optional: print full classification report for each batch size
for r in results_per_bs:
    print(f"\n--- Batch size {r['batch_size']} ---\n{r['classification_report']}")


--- Batch size 16 ---
              precision    recall  f1-score   support

      No PCL     0.9578    0.9462    0.9520      1895
         PCL     0.5405    0.6030    0.5701       199

    accuracy                         0.9136      2094
   macro avg     0.7492    0.7746    0.7610      2094
weighted avg     0.9181    0.9136    0.9157      2094


--- Batch size 32 ---
              precision    recall  f1-score   support

      No PCL     0.9597    0.9562    0.9580      1895
         PCL     0.5971    0.6181    0.6074       199

    accuracy                         0.9241      2094
   macro avg     0.7784    0.7871    0.7827      2094
weighted avg     0.9253    0.9241    0.9247      2094


--- Batch size 64 ---
              precision    recall  f1-score   support

      No PCL     0.9631    0.9377    0.9503      1895
         PCL     0.5261    0.6583    0.5848       199

    accuracy                         0.9112      2094
   macro avg     0.7446    0.7980    0.7675      2094
weigh

In [9]:
# Models and tokenizers were saved per batch size in the loop above.
# Paths: models/roberta_pcl_imbalance_weights_only_bs16, _bs32, _bs64

In [10]:
# Optional: produce test predictions using the best batch-size model
from torch.utils.data import DataLoader

test_df = pd.read_csv(
    RAW_DATA_DIR / "task4_test.tsv",
    sep="\t",
    names=["id", "par_id", "keyword", "country_code", "text"],
)
test_df["text"] = test_df["text"].astype(str).apply(strip_html_and_clean)
test_df["text"] = test_df["text"].replace("", " ")

best_model_dir = PROJECT_ROOT / "models" / f"roberta_pcl_imbalance_weights_only_bs{best_bs}"
tokenizer_bs = AutoTokenizer.from_pretrained(str(best_model_dir))
test_dataset_bs = PCLDataset(
    texts=test_df["text"],
    labels=np.zeros(len(test_df), dtype=int),
    tokenizer=tokenizer_bs,
    max_length=MAX_LENGTH,
)
model_bs = AutoModelForSequenceClassification.from_pretrained(str(best_model_dir))
model_bs.to(device)
model_bs.eval()
collator_bs = DataCollatorWithPadding(tokenizer=tokenizer_bs)
test_loader = DataLoader(test_dataset_bs, batch_size=best_bs, collate_fn=collator_bs)
all_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        out = model_bs(**batch)
        all_preds.extend(np.argmax(out.logits.cpu().numpy(), axis=-1).tolist())

test_txt_path = OUTPUT_DIR / f"test_bs{best_bs}.txt"
with open(test_txt_path, "w", encoding="utf-8") as f:
    for p in all_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions (best bs={best_bs}) to: {test_txt_path}")

Saved test predictions (best bs=32) to: /vol/bitbucket/tq422/NLP_cw/output/test_bs32.txt
